The original files are read-only inputs.

In [ ]:
# CareLoop Clinics — No-Show Guard
## Part 1: Python & Data Foundations (Tasks 1.1–1.6)

In [ ]:
## Setup
import csv
import json
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
APPOINTMENTS_FILE = DATA_DIR / "appointments.csv"
CLINICS_FILE = DATA_DIR / "clinics.csv"

# If running in Colab after uploading files to the root directory,
# change these paths to Path("appointments.csv") and Path("clinics.csv").
print("Appointments file exists:", APPOINTMENTS_FILE.exists())
print("Clinics file exists:", CLINICS_FILE.exists())

Appointments file exists: False
Clinics file exists: False


In [ ]:
## Task 1.1 — Read with csv.DictReader, convert types and reject conversion failures
with open(APPOINTMENTS_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    headers = reader.fieldnames
    raw_rows = list(reader)

print("Header:", headers)
print("Row count:", len(raw_rows))

def parse_row(row):
    converted = row.copy()
    converted["age"] = int(converted["age"])
    converted["fee_inr"] = int(converted["fee_inr"])
    converted["prior_no_shows"] = int(converted["prior_no_shows"])
    converted["distance_km"] = float(converted["distance_km"])
    return converted

accepted = []
rejected = []

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

In [ ]:
## Task 1.2 — Inspect and clean, pass 1
df = pd.read_csv(APPOINTMENTS_FILE)

print("Original shape:", df.shape)
print("\\nDataFrame info:")
df.info()
print("\\nMissing values:")
print(df.isnull().sum())
print("\\nExact duplicate rows:", df.duplicated().sum())
print("Raw booking_channel spellings:", df["booking_channel"].nunique())

df = df.drop_duplicates()
print("Rows after dropping exact duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after cleaning booking_channel:", len(df))
print("Distinct cleaned booking_channel values:", df["booking_channel"].nunique())
print("Cleaned labels:", sorted(df["booking_channel"].dropna().unique()))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

df = df.dropna(subset=["appt_date", "status"])
print("Rows after dropping missing/unreadable appt_date and missing status:", len(df))

In [ ]:
## Task 1.3 — Validity, gaps and outliers
# (a) Set impossible ages to missing.
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", int(invalid_age_mask.sum()))
df.loc[invalid_age_mask, "age"] = np.nan

# (b) Fill gaps with medians.
print("Missing ages before fill:", int(df["age"].isna().sum()))
print("Median age:", df["age"].median())
print("Median distance_km:", df["distance_km"].median())

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# (c) IQR fences and clipping.
def cap_outliers(series, column_name):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    capped_count = int(((series < lower) | (series > upper)).sum())
    print(f"{column_name}: Q1={q1}, Q3={q3}, IQR={iqr}, lower={lower}, upper={upper}")
    print(f"{column_name}: values capped={capped_count}")
    return np.clip(series, lower, upper)

df["distance_km"] = cap_outliers(df["distance_km"], "distance_km")
df["fee_inr"] = cap_outliers(df["fee_inr"], "fee_inr")

# (d) Required rounding.
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Largest fee after capping:", df["fee_inr"].max())

In [ ]:
## Task 1.4 — Features and NumPy summary

df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
column_means = np.mean(X, axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", column_means)

lead_mask = df["lead_days"] > 14
print("Rows with lead_days > 14:", int(lead_mask.sum()))
print("Share with lead_days > 14:", round(float(lead_mask.mean()) * 100, 1), "%")

distance_values = df["distance_km"].to_numpy()
z_scores = (distance_values - np.mean(distance_values)) / np.std(distance_values)
print("Rows with |z| > 2:", int((np.abs(z_scores) > 2).sum()))
print("No-shows:", int(df["no_show"].sum()), "out of", len(df))
print("No-show rate:", round(df["no_show"].mean() * 100, 1), "%")
print("Lead days min/max:", df["lead_days"].min(), df["lead_days"].max())

NameError: name 'df' is not defined

In [ ]:
## Task 1.5 — Enrich clinics using Open-Meteo API
 clinics = pd.read_csv(CLINICS_FILE)
temperatures = []

for _, clinic in clinics.iterrows():
    temp_c = None

    for attempt in range(2):  # first attempt + one retry
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": clinic["latitude"],
                    "longitude": clinic["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            if response.status_code != 200:
                last_reason = f"HTTP status {response.status_code}"
                if attempt == 0:
                    time.sleep(1)
                    continue
                break

            payload = response.json()
            hourly = payload.get("hourly", {})
            temperature_values = hourly.get("temperature_2m")

            if not temperature_values:
                last_reason = "hourly.temperature_2m missing or empty"
                if attempt == 0:
                    time.sleep(1)
                    continue
                break

            temp_c = temperature_values[0]
            last_reason = None
            break

        except requests.exceptions.RequestException as error:
            last_reason = str(error)
            if attempt == 0:
                time.sleep(1)

    if temp_c is None:
        print(f"Could not fetch temperature for {clinic['clinic_name']}: {last_reason}")

    temperatures.append(temp_c)

clinics["temp_c"] = temperatures
clinics.to_csv("clinics_enriched.csv", index=False)
print("Saved clinics_enriched.csv")
print(clinics[["clinic_id", "clinic_name", "temp_c"]])


In [ ]:
## Task 1.6 — Export clean appointments and generate SQL load script
required_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

df = df.sort_values("appointment_id").copy()
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
clean_df = df[required_columns].copy()

print("appointment_id unique:", clean_df["appointment_id"].is_unique)
print("Missing values by column:")
print(clean_df.isna().sum())
print("Total missing values:", int(clean_df.isna().sum().sum()))

if not clean_df["appointment_id"].is_unique:
    raise ValueError("appointment_id must be unique before export.")
if clean_df.isna().any().any():
    raise ValueError("Clean output still contains missing values. Inspect before export.")

clean_df.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean_df.shape)

# SQL literal helper: text is quoted, numeric values are unquoted, missing is NULL.
def sql_value(value, is_text=False):
    if pd.isna(value):
        return "NULL"
    if is_text:
        return "'" + str(value).replace("'", "''") + "'"
    return str(value)

clinic_columns = ["clinic_id", "clinic_name", "city", "tier", "latitude", "longitude", "temp_c"]
appointment_sql_text_columns = {
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "gender", "booking_channel", "visit_type", "reminder_sent", "status"
}

with open("load_data.sql", "w", encoding="utf-8") as sql_file:
    # Clinics first to satisfy the appointment-to-clinic foreign key.
    for _, row in clinics.iterrows():
        values = [
            sql_value(row[col], is_text=(col in {"clinic_id", "clinic_name", "city", "tier"}))
            for col in clinic_columns
        ]
        sql_file.write("INSERT INTO clinics VALUES (" + ",".join(values) + ");\\n")

    for _, row in clean_df.iterrows():
        values = [
            sql_value(row[col], is_text=(col in appointment_sql_text_columns))
            for col in required_columns
        ]
        sql_file.write("INSERT INTO appointments VALUES (" + ",".join(values) + ");\\n")

print("Saved load_data.sql")
print("Expected INSERT statements:", len(clinics) + len(clean_df))